# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/13aakash/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
# Setup — clone repo (Colab) and load data
import os, sys, subprocess
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/13aakash/flyrank-ml-internship.git"  # <-- check this is YOUR repo URL
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(raw):,} rows, {raw.shape[1]} columns")

30,000 rows, 44 columns


In [13]:
# Rebuild the honest feature vector — same steps you already used in Week 3
df = raw.copy()

numeric_fill_zero = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "trend_pct",
]
for col in numeric_fill_zero:
    df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

categorical_cols = [
    "competition_level", "content_type", "main_intent", "provider_used", "model_used",
    "age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier", "trend_direction",
]
for col in categorical_cols:
    df[col] = df[col].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

print(f"Prepared {len(df):,} rows | Declining rate: {df['is_declining_label'].mean():.3f}")

Prepared 30,000 rows | Declining rate: 0.542


In [14]:
# Import the already-vetted, leakage-safe feature lists (from your Week-3 work)
sys.path.insert(0, "scripts")
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES
print("Numeric:", MODEL_NUMERIC_FEATURES)
print("Categorical:", MODEL_CATEGORICAL_FEATURES)

Numeric: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Categorical: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']


In [15]:
# Rebuild the Week-4 baseline using only TRAIN clients to learn
# the position-bucket CTR thresholds, then apply them to all rows.

baseline_df = df.copy()

baseline_df["position_bucket"] = pd.cut(
    baseline_df["avg_position"],
    bins=[-np.inf, 5, 10, 20, 50, np.inf],
    labels=["1-5", "6-10", "11-20", "21-50", "50+"]
)

# FIT baseline thresholds on TRAIN rows only
train_bucket_medians = (
    baseline_df.iloc[train_idx]
    .groupby("position_bucket", observed=True)["ctr"]
    .median()
)

print("Position-bucket median CTR, learned from TRAIN clients only:")
print(train_bucket_medians)

# APPLY the train-derived thresholds to every row
baseline_df["position_bucket_median_ctr"] = (
    baseline_df["position_bucket"].map(train_bucket_medians)
)

baseline_df["weak_ctr_for_position"] = (
    baseline_df["ctr"] < baseline_df["position_bucket_median_ctr"]
).fillna(False).astype(int)

baseline_df["ctr_opportunity_points"] = (
    baseline_df["weak_ctr_for_position"] * 2
)

baseline_df["volume_points"] = np.select(
    [
        baseline_df["impressions_90d"] > 10000,
        baseline_df["impressions_90d"] > 1000
    ],
    [2, 1],
    default=0
)

baseline_df["action_score"] = (
    baseline_df["ctr_opportunity_points"]
    + baseline_df["volume_points"]
)

print("Baseline rebuilt:")
print(baseline_df["action_score"].value_counts().sort_index())

Position-bucket median CTR, learned from TRAIN clients only:
position_bucket
1-5      0.00
6-10     0.15
11-20    0.11
21-50    0.03
50+      0.00
Name: ctr, dtype: float64
Baseline rebuilt:
action_score
0     7884
1     7298
2    11626
3     2609
4      583
Name: count, dtype: int64


## 1. Method choice and why

My lane is Refresh / Content Opportunity Scoring, framed as a ranking/scoring problem
where Precision@K is the metric that matters (a content team can only review a limited
number of pages, so the top of the list has to be right).

I compare three methods, in increasing complexity, and let the numbers decide which is worth
keeping:

- **Logistic Regression** — simplest linear combination of my honest signals; the floor to beat.
- **Decision Tree (max_depth=5)** — captures non-linear thresholds ("weak CTR AND high volume")
  while staying mostly readable.
- **Random Forest (200 trees)** — an ensemble; the starter pipeline already showed this family
  beating a hand rule on this dataset, so I test it again on my own feature set.

I skip gradient boosting this week — the tree models above are already the natural next step
past my Week-4 rule.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Models to compare: logistic_regression, decision_tree, random_forest")
print("Baseline to beat: my Week-4 CTR-vs-position + search-volume rule")

Models to compare: logistic_regression, decision_tree, random_forest
Baseline to beat: my Week-4 CTR-vs-position + search-volume rule


## 2. Split design

I use a **client-grouped split** (`GroupShuffleSplit`, ~20% of clients held out) — not a random
row split. Reasons:

1. My Week-3 leakage hunt already showed a model can partly memorize `client_id` and look
   artificially strong on a random split. A grouped split tests the honest question: does this
   generalize to a client the model has never seen?
2. Only 32 clients exist; rows from the same client likely share house style and topics, so a
   random split would leak client-level patterns between train and test.

I don't use a time-aware split — the starter CSV is a single trailing-90-day snapshot with no
per-row date, so there's no timeline to walk forward on here.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"].astype(str).values
y = df["is_declining_label"].values

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df, y, groups=groups))

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])
print(f"Train rows: {len(train_idx):,} | Test rows: {len(test_idx):,}")
print(f"Train clients: {len(train_clients)} | Test clients: {len(test_clients)}")
print("Client overlap between train/test?", bool(train_clients & test_clients))

Train rows: 23,837 | Test rows: 6,163
Train clients: 25 | Test clients: 7
Client overlap between train/test? False


## 3. Train + compare vs my baseline

I evaluate all three models and my Week-4 baseline on the same held-out
client groups. The baseline's position-bucket CTR thresholds are learned
from the training clients only and then applied unchanged to the test
clients.

I use ROC-AUC and average precision as overall ranking metrics, but
Precision@20, Precision@50, and Precision@100 are the most useful metrics
for this lane because the content team can only review a limited number
of pages.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
numeric_frame = df[MODEL_NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = df[MODEL_CATEGORICAL_FEATURES].fillna("unknown").astype(str)
encoded_frame = pd.get_dummies(categorical_frame, prefix=MODEL_CATEGORICAL_FEATURES, dtype=float)
feature_frame = pd.concat([numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)], axis=1)
feature_columns = list(feature_frame.columns)

# Leakage sanity check
leakage_terms = ["trend", "declin", "flag", "health_score", "priority_score"]
assert not any(any(t in c.lower() for t in leakage_terms) for c in feature_columns), "Leaky column found!"
print(f"PASS: {len(feature_columns)} features, no leakage terms found.")

X_train, X_test = feature_frame.iloc[train_idx], feature_frame.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

PASS: 52 features, no leakage terms found.


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
    ]),
    "decision_tree": DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=42),
    "random_forest": RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=42),
}

test_probabilities = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    test_probabilities[name] = model.predict_proba(X_test)[:, 1]
    print(f"Trained {name}")

Trained logistic_regression
Trained decision_tree
Trained random_forest


In [20]:
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def metric_row(name, scores, labels):
    return {
        "method": name,
        "roc_auc": roc_auc_score(labels, scores),
        "avg_precision": average_precision_score(labels, scores),
        "precision_at_20": precision_at_k(scores, labels, 20),
        "precision_at_50": precision_at_k(scores, labels, 50),
        "precision_at_100": precision_at_k(scores, labels, 100),
    }

baseline_test_scores = baseline_df.iloc[test_idx]["action_score"].values
rows = [metric_row("baseline_rule (Week 4)", baseline_test_scores, y_test)]
for name, scores in test_probabilities.items():
    rows.append(metric_row(name, scores, y_test))

comparison_table = pd.DataFrame(rows).round(3)
comparison_table["base_rate"] = round(float(y_test.mean()), 3)
comparison_table

,method,roc_auc,avg_precision,precision_at_20,precision_at_50,precision_at_100,base_rate
0,baseline_rule (Week 4),0.517,0.525,0.50,0.56,0.53,0.511
1,logistic_regression,0.616,0.604,0.70,0.72,0.70,0.511
2,decision_tree,0.612,0.585,0.45,0.50,0.50,0.511
3,random_forest,0.610,0.590,0.50,0.54,0.60,0.511


## 4. Errors and interpretation

Logistic Regression was the strongest model in the client-holdout test:
it achieved Precision@20 of 0.70, Precision@50 of 0.72, and Precision@100
of 0.70, beating the Week-4 baseline at all three cutoffs. This suggests
the learned ranking is more useful for selecting a small review queue than
the hand-built rule, although the result is based on only seven held-out
clients and should not be treated as universal.

The strongest Random Forest signals were days_with_impressions,
log_impressions_90d, avg_position, and content_age_days. These are
plausible content-performance signals rather than target-derived fields,
and the leakage check found no prohibited feature terms.

The error cases suggest that very low-volume pages are difficult to rank:
some zero-CTR pages were false positives, while several pages with only
one or two impressions were false negatives. This indicates that sparse
performance data can make decline status difficult to distinguish reliably.

In [21]:
# Inspect the winning model: Logistic Regression
logistic_model = models["logistic_regression"]

logistic_coefficients = pd.Series(
    logistic_model.named_steps["model"].coef_[0],
    index=feature_columns
)

print("Top 10 features increasing predicted decline probability:")
print(
    logistic_coefficients
    .sort_values(ascending=False)
    .head(10)
)

print("\nTop 10 features decreasing predicted decline probability:")
print(
    logistic_coefficients
    .sort_values()
    .head(10)
)

Top 10 features increasing predicted decline probability:
log_impressions_90d             1.430558
word_count                      0.514848
impression_tier_low             0.301505
word_count_tier_1000-2000       0.253652
main_intent_unknown             0.190082
scroll_rate                     0.182084
days_since_last_update          0.159067
days_with_impressions           0.144368
content_type_keyword article    0.133272
position_tier_striking          0.075083
dtype: float64

Top 10 features decreasing predicted decline probability:
log_clicks_90d              -0.566999
avg_position                -0.385513
content_age_days            -0.344896
log_sessions_90d            -0.307488
char_count                  -0.302807
position_tier_top_3         -0.229653
impression_tier_good        -0.161437
impression_tier_excellent   -0.141701
competition_level_unknown   -0.136810
days_with_sessions          -0.123786
dtype: float64


In [22]:
test_frame["model_probability"] = test_probabilities["logistic_regression"]

false_positives = (
    test_frame[
        (test_frame["true_label"] == 0)
        & (test_frame["model_probability"] >= 0.6)
    ]
    .sort_values("model_probability", ascending=False)
    .head(3)
)

false_negatives = (
    test_frame[
        (test_frame["true_label"] == 1)
        & (test_frame["model_probability"] <= 0.4)
    ]
    .sort_values("model_probability")
    .head(3)
)

print("Confident false positives (model flagged decline, but page wasn't declining):")
display(false_positives)

print("\nConfident false negatives (model said safe, but page was declining):")
display(false_negatives)

Confident false positives (model flagged decline, but page wasn't declining):


,content_id,client_id,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,content_type,trend_direction,true_label,rf_probability,model_probability
26614,content_7be5f150dc65,client_f369cb89fc,290,5.9,0.0,96,20,keyword article,up,0,0.721560,0.945758
20736,content_41baf0722ad9,client_8527a891e2,3115,12.8,0.0,275,104,keyword article,stable,0,0.838642,0.932267
12869,content_5d5653c4eb4f,client_4e07408562,15101,5.7,0.0,421,7,keyword article,stable,0,0.437982,0.917224



Confident false negatives (model said safe, but page was declining):


,content_id,client_id,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,content_type,trend_direction,true_label,rf_probability,model_probability
8407,content_d1e915d03c28,client_4e07408562,2,45.0,0.0,537,104,keyword article,down,1,0.286633,0.050774
29158,content_e18144cbd19d,client_4e07408562,3,2.0,0.0,545,20,keyword article,down,1,0.350505,0.055044
17690,content_c268b1716236,client_e629fa6598,3,41.7,0.0,502,20,keyword article,down,1,0.285756,0.064743


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.